
## Patch Instruction for `AutoPipeline` Before Running MASE Pipeline

To prevent the `TypeError: object of type 'NoneType' has no len()` error, follow these steps:

1. **Open the file**: ./mase/src/chop/pipelines/auto_pipeline.py

2. **Locate line 26** inside the `AutoPipeline.__init__` method.

3. **Replace the existing line** with:

```python
self.pass_outputs = [{}] * len(self.pass_groups)

In [ ]:
!git clone --branch jtan/dev_physical https://github.com/tanjun8802/Mase_EDGE.git 
%cd /content
!rm -rf Mase_EDGE/mase
%cd Mase_EDGE
!git submodule update --init --recursive
%cd mase/
%pip install -e ".[executorch]"

In [ ]:
%cd /content/Mase_EDGE
!wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
!unzip tiny-imagenet-200.zip

In [ ]:
%cd /content/Mase_EDGE

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
import torchvision.datasets as datasets
import torchvision.models as models
from pathlib import Path
from copy import deepcopy
from PIL import Image

# ------------------------------
# 1️⃣ Device setup
# ------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

# device = "mps"

# Tiny ImageNet root (needs train/, val/images/, val/val_annotations.txt).
# If auto-detect fails, uncomment ONE of these instead of using resolve_tiny_imagenet_root():
# TINY_ROOT = Path("./tiny-imagenet-200")              # after !wget + !unzip next to notebook cwd (see ResNet18.ipynb)
# TINY_ROOT = Path("./mase/tiny-imagenet-200")        # cwd = repo root (ADLSProject)
# TINY_ROOT = Path("./../mase/tiny-imagenet-200")     # cwd = python notebooks/


def resolve_tiny_imagenet_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "tiny-imagenet-200",
        cwd.parent / "tiny-imagenet-200",  # e.g. cwd is .../mase after %cd mase/, zip unpacked in parent
        cwd / "mase" / "tiny-imagenet-200",
        cwd.parent / "mase" / "tiny-imagenet-200",
    ]
    for root in candidates:
        if (
            (root / "train").is_dir()
            and (root / "val" / "images").is_dir()
            and (root / "val" / "val_annotations.txt").is_file()
        ):
            return root
    raise FileNotFoundError(
        "Tiny ImageNet not found (expected train/, val/images/, val/val_annotations.txt). "
        f"cwd={cwd}. Uncomment a manual TINY_ROOT above or cd to the folder that contains mase/ or the zip extract."
    )


TINY_ROOT = resolve_tiny_imagenet_root()
print(f"Using Tiny ImageNet at: {TINY_ROOT}")

# ------------------------------
# 2️⃣ Tiny ImageNet dataloaders
# ------------------------------
# Train: labels from per-class subfolders (WordNet ids), same ordering as torchvision ImageFolder.
# Val: all images in val/images; labels from val_annotations.txt (filename → wnid), mapped with train's class_to_idx.
transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225])
])


class TinyImageNetValDataset(Dataset):
    def __init__(self, images_dir, annotations_path, class_to_idx, transform=None):
        self.images_dir = Path(images_dir)
        self.transform = transform
        self.samples = []
        with open(annotations_path) as f:
            for line in f:
                parts = line.strip().split("\t")
                if len(parts) < 2:
                    parts = line.strip().split()
                if len(parts) < 2:
                    continue
                fname, wnid = parts[0], parts[1]
                self.samples.append((self.images_dir / fname, class_to_idx[wnid]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, label


train_dataset = datasets.ImageFolder(root=TINY_ROOT / "train", transform=transform)
val_dataset = TinyImageNetValDataset(
    TINY_ROOT / "val" / "images",
    TINY_ROOT / "val" / "val_annotations.txt",
    train_dataset.class_to_idx,
    transform=transform,
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# ------------------------------
# 3️⃣ Model constructor (Tiny ImageNet head) — same idea as tutorial_6 `construct_model` / from_config
# ------------------------------
def mobilenet_v3_large_tiny_imagenet(num_classes: int = 200, pretrained: bool = True):
    weights = models.MobileNet_V3_Large_Weights.DEFAULT if pretrained else None
    m = models.mobilenet_v3_large(weights=weights)
    in_features = m.classifier[3].in_features
    m.classifier[3] = nn.Linear(in_features, num_classes)
    return m


model = mobilenet_v3_large_tiny_imagenet().to(device)

# ------------------------------
# 4️⃣ Loss + val helper (per-trial training happens in Optuna objective)
# ------------------------------
criterion = nn.CrossEntropyLoss()


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            _, preds = model(imgs).max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

acc_fp32_pretrained = evaluate(model, val_loader)
print(f"✅ FP32 val (pretrained head, no extra training): {acc_fp32_pretrained:.2f}%")

# ------------------------------
# 5️⃣ Mixed-precision CHOP PTQ (per-layer INT8 / FP16 / FP32 via quantize_transform_pass `by="name"`)
# ------------------------------
# FP32  → config name None (skip CHOP quant for that node)
# INT8  → integer 8-bit (activations/weights)
# FP16  → minifloat_ieee 16-bit, 5 exponent bits, bias 15 (IEEE FP16–like)

import optuna

from chop import MaseGraph
import chop.passes as passes
from chop.passes.graph.transforms.quantize import QUANTIZEABLE_OP
from chop.passes.graph.utils import get_mase_op

example_inputs = torch.randn(1, 3, 224, 224, device=device)
dummy_inputs = {"x": example_inputs}


def quant_cfg_fp32():
    return {"name": None}


def quant_cfg_int8():
    return {
        "name": "integer",
        "data_in_width": 8,
        "data_in_frac_width": 7,
        "weight_width": 8,
        "weight_frac_width": 7,
        "bias_width": 8,
        "bias_frac_width": 0,
    }


def quant_cfg_fp16():
    w, exp, eb = 16, 5, 15
    return {
        "name": "minifloat_ieee",
        "data_in_width": w,
        "data_in_exponent_width": exp,
        "data_in_exponent_bias": eb,
        "weight_width": w,
        "weight_exponent_width": exp,
        "weight_exponent_bias": eb,
        "bias_width": w,
        "bias_exponent_width": exp,
        "bias_exponent_bias": eb,
    }


LAYER_PRECISION_CHOICES = ("fp32", "int8", "fp16")


def choose_mixed_precision_scheme(trial, node_name: str, mase_op: str, module_target: str) -> str:
    """Per-layer categorical over INT8 / FP16 / FP32 (Optuna param = unique FX node name)."""
    if mase_op not in ("conv2d", "linear"):
        return "fp32"
    return trial.suggest_categorical(f"prec_{node_name}", LAYER_PRECISION_CHOICES)


def build_per_node_quantization_config(mg, trial):
    cfg = {"by": "name", "default": {"config": quant_cfg_fp32()}}
    for node in mg.fx_graph.nodes:
        op = get_mase_op(node)
        if op not in QUANTIZEABLE_OP:
            continue
        target = str(node.target) if node.op == "call_module" else ""
        label = choose_mixed_precision_scheme(trial, node.name, op, target)
        if label == "fp32":
            c = quant_cfg_fp32()
        elif label == "int8":
            c = quant_cfg_int8() if op in ("linear", "conv2d") else quant_cfg_fp32()
        elif label == "fp16":
            c = quant_cfg_fp16() if op in ("linear", "conv2d") else quant_cfg_fp32()
        else:
            raise ValueError(label)
        cfg[node.name] = {"config": c}
    return cfg


N_OPTUNA_TRIALS = 10
EPOCHS_PER_TRIAL = 1
BASE_SEED = 42


def train_one_epoch(module, optimizer_):
    module.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer_.zero_grad()
        loss = criterion(module(imgs), labels)
        loss.backward()
        optimizer_.step()
        running_loss += loss.item()
    return running_loss / len(train_loader)


def ptq_val_accuracy(finetuned_module, trial):
    """Fresh graph + mixed PTQ from `trial` suggestions; return val accuracy (%)."""
    model_ptq = mobilenet_v3_large_tiny_imagenet(pretrained=False).to(device)
    model_ptq.load_state_dict(finetuned_module.state_dict())
    model_ptq.eval()
    mg = MaseGraph(model_ptq)
    mg, _ = passes.init_metadata_analysis_pass(mg)
    mg, _ = passes.add_common_metadata_analysis_pass(mg, pass_args={"dummy_in": dummy_inputs})
    qcfg = build_per_node_quantization_config(mg, trial)
    mg, _ = passes.quantize_transform_pass(mg, pass_args=deepcopy(qcfg))
    ptq_model = torch.fx.GraphModule(mg.model, mg.fx_graph).to(device)
    return evaluate(ptq_model, val_loader)


def objective(trial):
    torch.manual_seed(BASE_SEED + trial.number)
    m = mobilenet_v3_large_tiny_imagenet().to(device)
    opt_t = optim.Adam(m.parameters(), lr=1e-4)
    for ep in range(EPOCHS_PER_TRIAL):
        loss_ep = train_one_epoch(m, opt_t)
    trial.set_user_attr("train_loss_last_epoch", loss_ep)
    return ptq_val_accuracy(m, trial)


quant_study = optuna.create_study(
    direction="maximize",
    study_name="mobilenetv3_mixed_ptq_mps",
)
quant_study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print(
    f"✅ Optuna done: {N_OPTUNA_TRIALS} trials × {EPOCHS_PER_TRIAL} train epoch(s) each; "
    f"best PTQ val = {quant_study.best_value:.2f}% (trial {quant_study.best_trial.number})"
)

# Replay best trial: same seed + 1 epoch → `model` for QAT / export; confirm PTQ with fixed params
torch.manual_seed(BASE_SEED + quant_study.best_trial.number)
model = mobilenet_v3_large_tiny_imagenet().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
for _ in range(EPOCHS_PER_TRIAL):
    train_one_epoch(model, optimizer)
acc_fp32 = evaluate(model, val_loader)
print(f"✅ FP32 val after replay (best-trial seed, {EPOCHS_PER_TRIAL} epoch): {acc_fp32:.2f}%")

acc_ptq = ptq_val_accuracy(model, optuna.trial.FixedTrial(quant_study.best_trial.params))
print(f"✅ Best-layout PTQ val (replay): {acc_ptq:.2f}%")

# ------------------------------
# 6️⃣ Optional: QAT-like fine-tuning (PyTorch training)
# ------------------------------
# Perform QAT-like training on the FP32 model instead of FX Graph with IntegerQuantize
qat_model = model  # use FP32 model for export
optimizer = optim.Adam(qat_model.parameters(), lr=1e-4)
epochs_qat = 5  # short demo

for epoch in range(epochs_qat):
    qat_model.train()
    running_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(qat_model(imgs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"QAT Epoch {epoch+1}/{epochs_qat}, Loss: {running_loss/len(train_loader):.4f}")

acc_qat = evaluate(qat_model, val_loader)
print(f"✅ QAT Accuracy: {acc_qat:.2f}%")

# ------------------------------
# 7️⃣ Export FP32 QAT weights + best Optuna PTQ model to .pt
# ------------------------------
output_dir = Path("./mase_output")
output_dir.mkdir(exist_ok=True)

qat_model.eval()
pt_path = output_dir / "mobilenet_v3_large_qat_fp32.pt"
checkpoint = {
    "state_dict": {k: v.detach().cpu() for k, v in qat_model.state_dict().items()},
    "num_classes": 200,
}
torch.save(checkpoint, pt_path)
print(f"✅ PyTorch checkpoint saved to {pt_path}")
print("   Load with: m = mobilenet_v3_large_tiny_imagenet(pretrained=False); m.load_state_dict(torch.load(pt_path, map_location='cpu')['state_dict'])")

# Best mixed-precision PTQ model (Optuna best trial) — full FX GraphModule
_fixed_trial = optuna.trial.FixedTrial(dict(quant_study.best_trial.params))
_best_ptq_base = mobilenet_v3_large_tiny_imagenet(pretrained=False).to(device)
_best_ptq_base.load_state_dict(model.state_dict())
_best_ptq_base.eval()
_mg_best = MaseGraph(_best_ptq_base)
_mg_best, _ = passes.init_metadata_analysis_pass(_mg_best)
_mg_best, _ = passes.add_common_metadata_analysis_pass(_mg_best, pass_args={"dummy_in": dummy_inputs})
_qcfg_best = build_per_node_quantization_config(_mg_best, _fixed_trial)
_mg_best, _ = passes.quantize_transform_pass(_mg_best, pass_args=deepcopy(_qcfg_best))
best_ptq_model = torch.fx.GraphModule(_mg_best.model, _mg_best.fx_graph).eval().cpu()
ptq_path = output_dir / "mobilenet_v3_best_ptq.pt"
torch.save(best_ptq_model, ptq_path)
print(f"✅ Best PTQ model (trial {quant_study.best_trial.number}, val={quant_study.best_value:.2f}%) saved to {ptq_path}")
print("   Load with: m = torch.load(ptq_path, map_location='cpu'); m.eval()")

try:
    from google.colab import files
    files.download(str(ptq_path))
except ImportError:
    try:
        from IPython.display import FileLink, display
        display(FileLink(ptq_path, result_html_prefix="Download best PTQ .pt: "))
    except ImportError:
        print(f"   Download manually from: {ptq_path.resolve()}")


Using Tiny ImageNet at: /content/Mase_EDGE/tiny-imagenet-200


[I 2026-03-26 17:04:13,165] A new study created in memory with name: mobilenetv3_mixed_ptq_mps


✅ FP32 val (pretrained head, no extra training): 0.54%


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-03-26 17:08:22,480] Trial 0 finished with value: 0.5 and parameters: {'prec_features_0_0': 'fp32', 'prec_features_1_block_0_0': 'fp32', 'prec_features_1_block_1_0': 'fp32', 'prec_features_2_block_0_0': 'fp16', 'prec_features_2_block_1_0': 'int8', 'prec_features_2_block_2_0': 'int8', 'prec_features_3_block_0_0': 'fp16', 'prec_features_3_block_1_0': 'fp32', 'prec_features_3_block_2_0': 'fp16', 'prec_features_4_block_0_0': 'fp16', 'prec_features_4_block_1_0': 'fp16', 'prec_features_4_block_2_fc1': 'fp32', 'prec_features_4_block_2_fc2': 'int8', 'prec_features_4_block_3_0': 'fp16', 'prec_features_5_block_0_0': 'fp32', 'prec_features_5_block_1_0': 'fp16', 'prec_features_5_block_2_fc1': 'fp32', 'prec_features_5_block_2_fc2': 'fp16', 'prec_features_5_block_3_0': 'fp32', 'prec_features_6_block_0_0': 'fp16', 'prec_features_6_block_1_0': 'fp16', 'prec_features_6_block_2_fc1': 'fp16', 'prec_features_6_block_2_fc2': 'fp32', 'prec_features_6_block_3_0': 'fp16', 'prec_features_7_block_0_0': '

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# Run the training cell above first so `model` and `device` exist.
# Installs torchinfo into the active Jupyter kernel if missing.
%pip install -q torchinfo

from torchinfo import summary

summary(
    model,
    input_size=(1, 3, 224, 224),
    col_names=("input_size", "output_size", "num_params"),
    device=device,
)

Layer (type:depth-idx)                             Input Shape               Output Shape              Param #
MobileNetV3                                        [1, 3, 224, 224]          [1, 200]                  --
├─Sequential: 1-1                                  [1, 3, 224, 224]          [1, 960, 7, 7]            --
│    └─Conv2dNormActivation: 2-1                   [1, 3, 224, 224]          [1, 16, 112, 112]         --
│    │    └─Conv2d: 3-1                            [1, 3, 224, 224]          [1, 16, 112, 112]         432
│    │    └─BatchNorm2d: 3-2                       [1, 16, 112, 112]         [1, 16, 112, 112]         32
│    │    └─Hardswish: 3-3                         [1, 16, 112, 112]         [1, 16, 112, 112]         --
│    └─InvertedResidual: 2-2                       [1, 16, 112, 112]         [1, 16, 112, 112]         --
│    │    └─Sequential: 3-4                        [1, 16, 112, 112]         [1, 16, 112, 112]         464
│    └─InvertedResidual: 2-3           

In [8]:
# Run after the main training/export cell. Colab: mount Drive and copy `.pt` checkpoints to MyDrive.
from pathlib import Path
import shutil

_output = Path("./mase_output")
_pt_files = [
    _output / "mobilenet_v3_large_qat_fp32.pt",
    _output / "mobilenet_v3_best_ptq.pt",
]

try:
    from google.colab import drive
except ImportError:
    print("Not in Google Colab — skipping Drive mount. Checkpoints stay in:", _output.resolve())
else:
    drive.mount("/content/drive", force_remount=False)
    _drive_dir = Path("/content/drive/MyDrive/MobileNetV3_Exports")
    _drive_dir.mkdir(parents=True, exist_ok=True)
    for _p in _pt_files:
        if _p.is_file():
            _dest = _drive_dir / _p.name
            shutil.copy2(_p, _dest)
            print(f"Saved to Drive: {_dest}")
        else:
            print(f"Missing (run export cell first): {_p}")


Mounted at /content/drive
Saved to Drive: /content/drive/MyDrive/MobileNetV3_Exports/mobilenet_v3_large_qat_fp32.pt
Saved to Drive: /content/drive/MyDrive/MobileNetV3_Exports/mobilenet_v3_best_ptq.pt
